# Dental Vision V1 — DENTEX diagnostic training
T4 workflow. This version deliberately selects the DENTEX diagnostic annotation set (caries, deep caries, periapical lesion, impacted tooth), not the quadrant set. It also copies the final checkpoint to Google Drive.


In [ ]:
!nvidia-smi


In [ ]:
import os
if not os.path.exists('/content/dental-vision-v1'):
    !git clone https://github.com/drhaidarali95/dental-vision-v1.git /content/dental-vision-v1
%cd /content/dental-vision-v1
!git pull
!pip -q install -r requirements.txt


In [ ]:
!python scripts/download_dentex.py --out data/dentex


In [ ]:
import pathlib, zipfile
root=pathlib.Path('data/dentex')
for z in root.glob('*.zip'):
    dest=root/z.stem
    dest.mkdir(parents=True,exist_ok=True)
    marker=dest/'.extracted'
    if not marker.exists():
        print('Extracting',z,'->',dest)
        with zipfile.ZipFile(z) as f: f.extractall(dest)
        marker.touch()
    else: print('Already extracted:',z)
!python scripts/inspect_dentex.py data/dentex


In [ ]:
import json, pathlib
root=pathlib.Path('data/dentex')
wanted={'caries','deep caries','periapical lesion','periapical lesions','impacted tooth','impacted teeth'}
candidates=[]
for p in root.rglob('*.json'):
    try: d=json.loads(p.read_text())
    except Exception: continue
    if not isinstance(d,dict) or not {'images','annotations','categories'}.issubset(d): continue
    names={str(c.get('name','')).strip().lower() for c in d['categories']}
    score=len(names & wanted)
    print('COCO:',p,'images=',len(d['images']),'annotations=',len(d['annotations']),'categories=',sorted(names),'diagnostic_score=',score)
    if score: candidates.append((score,len(d['images']),p,d))
assert candidates, 'No diagnostic COCO JSON found. Do NOT fall back to quadrant training.'
score,n,ann,d=max(candidates,key=lambda x:(x[0],x[1]))
names={str(c.get('name','')).strip().lower() for c in d['categories']}
assert score>=3, f'Found only {score} expected diagnostic classes: {names}'
print('SELECTED DIAGNOSTIC ANNOTATIONS:',ann)
print('SELECTED CATEGORIES:',[(c.get('id'),c.get('name')) for c in d['categories']])


## Train diagnostic detector
The next cell resolves the actual image root from the selected diagnostic COCO file and trains 20 epochs.


In [ ]:
import subprocess,sys,pathlib
sample=d['images'][0]['file_name']
matches=list(root.rglob(pathlib.Path(sample).name))
assert matches,f'Could not locate sample image {sample}'
img_path=matches[0]
image_root=img_path
for _ in pathlib.Path(sample).parts: image_root=image_root.parent
out='checkpoints/dentex_diagnostic_v1.pt'
print('Image root:',image_root)
cmd=[sys.executable,'train.py','--images',str(image_root),'--annotations',str(ann),'--epochs','20','--batch-size','2','--output',out]
print('Launching diagnostic training:', ' '.join(cmd))
subprocess.run(cmd,check=True)
assert pathlib.Path(out).exists(),'Training ended without checkpoint'
print('CHECKPOINT READY:',out,'MB=',round(pathlib.Path(out).stat().st_size/1024/1024,1))


## Persist checkpoint
Colab runtimes are temporary. Mount Drive and copy the trained weights there.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pathlib,shutil
src=pathlib.Path('checkpoints/dentex_diagnostic_v1.pt')
dst=pathlib.Path('/content/drive/MyDrive/DentalVisionV1/dentex_diagnostic_v1.pt')
dst.parent.mkdir(parents=True,exist_ok=True)
shutil.copy2(src,dst)
print('SAVED PERMANENTLY:',dst)
